In [1]:
pip install psycopg2-binary pyserial schedule


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Python backend
import os
import psycopg2
import serial
import time
import threading
import schedule
from datetime import datetime, timedelta
import smtplib
from email.message import EmailMessage
import ssl

# PostgreSQL config
DB_CONFIG = {
    'dbname': 'vendingdb',
    'user': 'postgres',
    'password': 'yourpassword',
    'host': 'localhost',
    'port': '5432'
}

# Serial config (auto-detect the correct Pi serial port)
def resolve_serial_port():
    for port in ['/dev/serial0', '/dev/ttyAMA0', '/dev/ttyS0', '/dev/ttyUSB0', '/dev/ttyACM0']:
        if os.path.exists(port):
            return port
    return None

serial_port = resolve_serial_port()
if serial_port is None:
    raise RuntimeError('No supported serial port was detected for the Raspberry Pi hardware.')
arduino = serial.Serial(serial_port, 9600, timeout=1)
time.sleep(2)  # Allow hardware to reset

# DB Connection
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()

# LCD Commands to Arduino
def lcd_message(message):
    arduino.write(f"LCD:{message}\n".encode())

# Dispense command
def dispense(item):
    arduino.write(f"DISPENSE:{item}\n".encode())

# Accept return command
def accept_return(item):
    arduino.write(f"RETURN:{item}\n".encode())

# Transaction logging
def log_transaction(student_id, name, email, item):
    cursor.execute("""
        INSERT INTO transactions (student_id, student_name, email, item)
        VALUES (%s, %s, %s, %s)
        RETURNING id
    """, (student_id, name, email, item))
    conn.commit()
    lcd_message(f"{name}, {item} out")
    dispense(item)
    send_email(email, "Equipment Checked Out", f"You borrowed {item} at {datetime.now().strftime('%H:%M')}")
    threading.Thread(target=reminder_timer, args=(student_id, item, email)).start()

def return_item(student_id):
    cursor.execute("""
        SELECT id, item FROM transactions
        WHERE student_id = %s AND returned = FALSE
        ORDER BY checkout_time DESC LIMIT 1
    """, (student_id,))
    row = cursor.fetchone()
    if row:
        tid, item = row
        cursor.execute("""
            UPDATE transactions SET returned = TRUE, return_time = NOW()
            WHERE id = %s
        """, (tid,))
        conn.commit()
        accept_return(item)
        lcd_message("Return accepted")
    else:
        lcd_message("Nothing to return")

# 2-hour reminder
def reminder_timer(student_id, item, email):
    time.sleep(2 * 60 * 60)
    cursor.execute("""
        SELECT returned FROM transactions
        WHERE student_id = %s AND item = %s ORDER BY checkout_time DESC LIMIT 1
    """, (student_id, item))
    returned = cursor.fetchone()
    if returned and not returned[0]:
        send_email(email, "Return Reminder", f"Please return {item} ASAP.")

# End-of-day escalation
def end_of_day_check():
    cursor.execute("""
        SELECT student_name, student_id, email, item FROM transactions
        WHERE returned = FALSE AND checkout_time::date = CURRENT_DATE
    """)
    for name, sid, email, item in cursor.fetchall():
        send_email(email, "Final Warning", f"You did not return {item} today.")
        send_email("studentservices@school.edu", "Unreturned Equipment Alert",
                   f"{name} ({sid}) has not returned {item}.")

schedule.every().day.at("23:55").do(end_of_day_check)

SENDER_EMAIL = "your_email@gmail.com"
PASSWORD = "your_app_password"

def send_email(to, subject, body):
    msg = EmailMessage()
    msg['Subject'] = subject
    msg['From'] = SENDER_EMAIL
    msg['To'] = to
    msg.set_content(body)

    context = ssl.create_default_context()
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
        server.login(SENDER_EMAIL, PASSWORD)
        server.send_message(msg)

# MAIN LOOP
def run():
    lcd_message("Ready. Scan ID.")
    while True:
        if arduino.in_waiting > 0:
            line = arduino.readline().decode().strip()
            if line.startswith("SCAN:"):
                student_id = line.split(":")[1]
                lcd_message("Enter 1=Out, 2=Return")
                mode = arduino.readline().decode().strip()
                if mode == "1":
                    arduino.write(b"GETINFO\n")  # Trigger Arduino to send student name/email
                    name = arduino.readline().decode().strip().replace("NAME:", "")
                    email = arduino.readline().decode().strip().replace("EMAIL:", "")
                    lcd_message("Item?")
                    item = arduino.readline().decode().strip().replace("ITEM:", "")
                    log_transaction(student_id, name, email, item)
                elif mode == "2":
                    return_item(student_id)
                else:
                    lcd_message("Invalid mode")
        schedule.run_pending()
        time.sleep(0.5)

if __name__ == "__main__":
    run()

SerialException: could not open port '/dev/ttyUSB0': FileNotFoundError(2, 'The system cannot find the path specified.', None, 3)